# HANCOCK: intermediate-fusion survival

Needs the HANCOCK data under `data/hancock/` and the TabICL checkpoint; without both, see
`examples/synthetic/synthetic_survival.ipynb` instead.

This notebook mirrors `survival_intermediate.py`, defaulting with HANCOCK's 'in' split.

In [ ]:
from functools import partial
from pathlib import Path

import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from kalecancer.evaluate import HarrellC, UnoC
from kalecancer.loaddata import MultimodalDataset, PatchFeatures, TimeToEvent
from kalecancer.model import ABMIL, Concat, CoxHead, IntermediateFusion, TabICLEncoder
from kalecancer.pipeline import EarlyStopping, Pipeline

try:
    HERE = Path(__file__).resolve().parent
except NameError:  # __file__ is undefined when this cell runs inside a notebook kernel
    HERE = Path.cwd()

DATA = HERE.parents[1] / "data" / "hancock"

BASELINE_COLUMNS = ["age_at_initial_diagnosis", "sex", "smoking_status", "primarily_metastasis"]

SPLIT_NAME = "in"  # "in", "out", "Oropharynx".

## 1. Data

In [ ]:
def official_split(data: MultimodalDataset, name: str) -> tuple[MultimodalDataset, MultimodalDataset]:
    """The published training/test assignment, restricted to the patients the dataset includes.

    Names: "in", "out", "Oropharynx". The "treatment_outcome" test set is selected on outcome (76% events), so it
    is not a fair test set for overall survival.
    """
    path = DATA / "DataSplits_DataDictionaries" / f"dataset_split_{name}.json"
    split = pd.read_json(path, dtype={"patient_id": str}).set_index("patient_id")["dataset"]
    split = split[split.index.isin(data.ids)]
    return data.subset(split.index[split == "training"]), data.subset(split.index[split == "test"])

### 1.1 Loading the data from file

In [ ]:
table = pd.read_json(DATA / "StructuredData" / "clinical_data.json", dtype={"patient_id": str}).set_index("patient_id")

clinical = table[BASELINE_COLUMNS]

target = TimeToEvent(
    time=table["days_to_last_information"],
    event=table["survival_status"].map({"deceased": True, "living": False})
)

wsi = PatchFeatures.from_glob(
    str(DATA / "WSI_PrimaryTumor_UNI_Encodings" / "*" / "h5_files" / "*.h5"),
    id_pattern=r"PrimaryTumor_HE_(\d{3})(?:_[a-z])?\.h5$",
    multiple_files="concatenate",
)


### 1.2 Creating the multimodal dataset

In [ ]:
dataset =  MultimodalDataset(
    modalities={
        "clinical": clinical, 
        "wsi": wsi
    },
    target=target,
    required_modalities=["clinical", "wsi"] # only patients with both modalities will be kept
)

In [ ]:
dataset.summary()

,count
available: clinical,763
available: wsi,701
available: target,763
included,701
included with clinical,701
included with wsi,701
excluded: missing required modality 'wsi',62
events,192


In [ ]:
# get the official HANCOCK train/test split
train, test = official_split(dataset, SPLIT_NAME)

## 2. Model and pipeline

### 2.1 Define how the tabular data columns should be transformed

In [ ]:
categorical_transforms = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="missing"),
    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
)

tabular_transforms = ColumnTransformer([
    ("numeric", StandardScaler(), ["age_at_initial_diagnosis"]),
    ("categorical", categorical_transforms, ["sex", "smoking_status", "primarily_metastasis"]),
])

### 2.2 define the fusion model

In [ ]:
# intermediate fusion = fuse after encoding
model = IntermediateFusion(

    # how each modality should be encoded
    encoding={
        "clinical": [
            TabICLEncoder(
                checkpoint="tabicl-classifier-v2-20260212.ckpt",
                output="row",
                trainable=[],
                context_label="event",
                context_folds=5,
                random_state=0,
            ),
            torch.nn.Linear(512, 64),
        ],

        "wsi": [
            ABMIL(in_dim=1024, hidden_dim=256, attention_dim=128, dropout=0.25),
            torch.nn.Linear(256, 64),
        ],

    },

    # how to fuse the encoded modalities
    fusion=Concat(),

    # the prediction head to use after the fusion
    head=CoxHead(in_dim=128, ties="efron"),
)

### 2.3 Build the pipeline that will run the leakage-free training loop

In [ ]:
pipe = Pipeline(
    model=model,
    transforms={"clinical": tabular_transforms},
    optimizer=partial(torch.optim.AdamW, lr=3e-4, weight_decay=1e-2),
    batch_size=32,
    drop_last=True,
    max_epochs=30,
    validation=StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=0),
    early_stopping=EarlyStopping(metric=HarrellC(), patience=5, restore_best=True),
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision="32-true",
    random_state=0,
)

## 3. Train and evaluate

### 3.1 Training

In [ ]:
pipe.fit(train)

,model,IntermediateF...s=False) ) )
,transforms,{'clinical': ColumnTransfo...etastasis'])])}
,optimizer,functools.par...ht_decay=0.01)
,batch_size,32
,drop_last,True
,max_epochs,30
,validation,StratifiedShu...ain_size=None)
,early_stopping,<kalecancer.p...x7aa724732cf0>
,accelerator,'gpu'
,precision,'32-true'
,random_state,0


In [ ]:
pipe.history_

,train_loss,val_loss,val_metric,optimizer_steps,skipped_batches
epoch,,,,,
0,2.737619,3.794892,0.527592,14,0
1,2.548712,3.798853,0.544314,28,0
2,2.437233,3.868830,0.566054,42,0
3,2.188779,3.973521,0.569398,56,0
4,2.169811,3.971967,0.595318,70,0
5,2.061028,3.988656,0.583612,84,0
6,1.948105,4.214475,0.580268,98,0
7,1.879060,4.383988,0.581104,112,0
8,1.839110,4.619885,0.567726,126,0


In [ ]:
pipe.fit_report_

{'n_train': 471,
 'n_val': 84,
 'events': 130,
 'epochs_run': 10,
 'best_epoch': 4,
 'best_score': 0.5953177213668823,
 'optimizer_steps': 140,
 'skipped_batches': 0}

### 3.2 Evaluation

In [ ]:
pipe.evaluate(test, metrics={"harrell_c": HarrellC(), "uno_c_5y": UnoC(tau=1825.0)})

harrell_c    0.687148
uno_c_5y     0.699938
Name: score, dtype: float64